# Semantic Model Validations

Live demo notebook for Semantic Model Maintenance focused on Best Practice Analyzer (BPA) and Vertipaq.

In [ ]:
%pip install -U semantic-link-labs

In [ ]:
import sempy_labs as labs
import pandas as pd

## What Is BPA?

- Best Practice Analyzer (BPA) — originally created by Daniel Otykier for Tabular Editor - was brought natively into Python via Semantic Link Labs
- BPA evaluates semantic model objects (tables, columns, measures, relationships, and metadata) against a rules table and returns violations.

In [ ]:
# Update for your demo
workspace_name = "Demo"
dataset_name = "card_udf"

In [ ]:

# Run BPA with the standard built-in rules
bpa_default_df = labs.run_model_bpa(
    dataset=dataset_name,
    workspace=workspace_name
)

display(bpa_default_df)

In [ ]:
bpa_default_df = labs.run_model_bpa(
    dataset=dataset_name,
    workspace=workspace_name,
    return_dataframe=True
)

display(bpa_default_df.head(20))
print(f"Total violations: {len(bpa_default_df)}")

## Custom Rule Format

Custom rules are passed as a pandas DataFrame with these columns:

- Category
- Scope
- Severity
- Rule Name
- Expression
- Description
- URL

| Column | Allowed Options | Notes |
|---|---|---|
| Severity | `Error`, `Warning`, `Info` | Fixed set |
| Scope | `Model`, `Table`, `Calculated Table`, `Column`, `Calculated Column`, `Measure`, `Hierarchy`, `Relationship`, `Role`, `Row Level Security`, `Calculation Item`, `Partition`, `Function` | Can be a single value or a list |
| Category | Any text | Free-form |
| Rule Name | Any text | Free-form |
| Description | Any text | Free-form |
| URL | Any text/URL | Free-form |
| Expression | Python lambda | Returns `True` when object violates the rule |

### How the Expression Works

The Expression column contains the rule logic used by BPA to evaluate each semantic model object in scope.

Each expression is usually a Python lambda with the signature `lambda obj, tom: <condition>`.

- obj is the current object being evaluated (for example a table, column, or measure, based on Scope).
- tom is the connected semantic model context (Tabular Object Model) that lets you inspect broader model metadata.

The expression must return a boolean:
- True: the object violates the rule and appears in BPA results.
- False: the object passes the rule.

Think of it as a filter for anti-patterns: if the condition detects a problem, return True.

Examples:
- `lambda obj, tom: " " in obj.Name`
- `lambda obj, tom: obj.Name[:1].islower()`
- `lambda obj, tom: obj.IsHidden`

Keep expressions deterministic and lightweight so scans remain fast and predictable.


In [ ]:
rules = pd.DataFrame(
    [
        (
            "Maintainability",
            "Table",
            "Info",
            "Avoid spaces in table names",
            lambda obj, tom: " " in obj.Name,
            "Table names without spaces are easier to reference and automate against.",
            ""
        ),
        (
            "Maintainability",
            "Measure",
            "Info",
            "Avoid lowercase-first measure names",
            lambda obj, tom: obj.Name[0] != obj.Name[0].upper(),
            "Using uppercase-first measure names improves consistency and readability.",
            ""
        ),
        (
            "Naming",
            ["Table", "Column"],
            "Info",
            "First letter of objects must be capitalized",
            lambda obj, tom: obj.Name[0] != obj.Name[0].upper(),
            "The first letter of object names should be capitalized to maintain professional quality.",
            ""
        )
    ],
    columns=[
        "Category",
        "Scope",
        "Severity",
        "Rule Name",
        "Expression",
        "Description",
        "URL"
    ]
)

bpa_custom_df = labs.run_model_bpa(
    dataset=dataset_name,
    workspace=workspace_name,
    rules=rules,
)

display(bpa_custom_df)

## Run BPA In Bulk

`labs.run_model_bpa_bulk(...)` scans all semantic models in one or more workspaces and appends results to `modelbparesults` in the attached lakehouse.

Notes:
- A lakehouse must be attached to the notebook.
- You can pass built-in rules (default) or your custom `rules` DataFrame.
- You can skip noisy/default models using `skip_models` and `skip_models_in_workspace`.

In [ ]:
#Run in Bulk
labs.run_model_bpa_bulk(
    workspace="Demo",
    rules=rules,
    skip_models=["ModelBPA", "Fabric Capacity Metrics"],
)

In [ ]:
%%tsql -artifact Lakehouse -type Lakehouse
SELECT *
FROM dbo.modelbparesults

## What Is VertiPaq Analyzer?

VertiPaq Analyzer helps you understand where memory is being used inside a semantic model.
It breaks down model size by tables, columns, relationships, and hierarchies so you can quickly spot heavy objects and optimize them for better performance and capacity efficiency.

In [ ]:
# VertiPaq Analyzer (interactive view)
labs.vertipaq_analyzer(
    dataset=dataset_name,
    workspace=workspace_name
)

In [ ]:
target_lakehouse = "Lakehouse"
target_lakehouse_workspace = "Demo"

labs.vertipaq_analyzer(
    dataset=dataset_name,
    workspace=workspace_name,
    export="table",
    export_lakehouse=target_lakehouse,
    export_workspace=target_lakehouse_workspace
    # export_schema="dbo",  # Optional when lakehouse schemas are enabled
)

In [ ]:
%%tsql -artifact Lakehouse -type Lakehouse
SELECT *
FROM vertipaqanalyzer_columns

## What Are Annotations?

Annotations are metadata properties stored on semantic model objects (for example tables, columns, relationships, and partitions).

They are useful for storing helper statistics and context that rules can read during analysis.
In BPA scenarios, VertiPaq-related annotations can hold values like row count or object size for advanced checks.

If you do not use annotation-based rules, standard BPA rules run normally without them.

In [ ]:
# Custom BPA rules using built-in VertiPaq annotation helpers (no custom method)
import pandas as pd

rules_vertipaq = pd.DataFrame(
    [
        (
            "VertiPaq",
            "Table",
            "Warning",
            "Large table total size",
            lambda obj, tom: tom.total_size(obj) > 50_000_000,
            "Flags tables with VertiPaq total size over 50 MB (50,000,000 bytes).",
            ""
        ),
        (
            "VertiPaq",
            "Table",
            "Info",
            "High row count table",
            lambda obj, tom: tom.row_count(obj) > 50_000,
            "Flags tables with more than 5 million rows based on VertiPaq annotations.",
            ""
        )
    ],
    columns=[
        "Category",
        "Scope",
        "Severity",
        "Rule Name",
        "Expression",
        "Description",
        "URL"
    ]
)

# Important: extended=True populates VertiPaq annotations before rules are evaluated
bpa_vertipaq_rules_df = labs.run_model_bpa(
    dataset=dataset_name,
    workspace=workspace_name,
    rules=rules_vertipaq,
    extended=True, #tells BPA to populate VertiPaq statistics,
    return_dataframe=True
)

display(bpa_vertipaq_rules_df)

# Clear Annotations

In [ ]:
from sempy_labs.tom import connect_semantic_model

def clear_vertipaq_annotations(dataset: str, workspace: str | None = None):
    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=False) as tom:
        tom.remove_vertipaq_annotations()
    print(f"Cleared VertiPaq annotations for '{dataset}'.")